# FDA Drug Safety Signal Detection — Exploratory Analysis

This notebook walks through the pipeline end-to-end on a single quarter of FAERS data, so you can sanity-check each layer before running the full 10M-row job.

**Author:** Saif Mohammed · MSCSDS, Seattle University · smohammed8@seattleu.edu  
**Project:** [github.com/Mohammed-Saif-07](https://github.com/Mohammed-Saif-07)

## Outline
1. Inspect a raw FAERS JSON file
2. Look at the Parquet output
3. Reproduce PRR / ROR on a sample, compare to the Hive table
4. Inspect detected signals + cross-reference against `fda_warnings.csv`
5. Visualise top signals and the backtest result

In [ ]:
import os, json, sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
print('Project root:', ROOT)

## 1. Inspect a raw FAERS JSON file
We expect a top-level `results` array, each entry having `patient.drug[]` and `patient.reaction[]`.

In [ ]:
import zipfile
raw_dir = ROOT / 'data' / 'raw'
candidates = sorted(raw_dir.rglob('drug-event-*.json.zip'))
print(f'Found {len(candidates)} raw FAERS files')
if candidates:
    f = candidates[0]
    print('Opening', f)
    with zipfile.ZipFile(f) as z:
        with z.open(z.namelist()[0]) as inner:
            doc = json.load(inner)
    print('top keys:', list(doc.keys()))
    print('first report keys:', list(doc['results'][0].keys())[:10])
    print('drugs in report 0:', [d.get('medicinalproduct') for d in doc['results'][0]['patient']['drug']])

## 2. Look at the Parquet output
After running `spark-submit ingestion/parse_faers.py`.

In [ ]:
proc = ROOT / 'data' / 'processed' / 'raw_adverse_events'
if proc.exists():
    df = pd.read_parquet(proc)
    print(f'{len(df):,} rows')
    df.head()

## 3. Reproduce PRR / ROR in pandas on a sample
Same formula as `hive/signal_detection.hql` so we can verify the Hive output matches.

In [ ]:
def prr_ror(df):
    pair = (df.groupby(['drug_name','reaction_term'])
              .agg(a=('safetyreportid','nunique')).reset_index())
    drug_total = pair.groupby('drug_name')['a'].sum().to_dict()
    rx_total   = pair.groupby('reaction_term')['a'].sum().to_dict()
    N          = pair['a'].sum()
    pair['drug_total'] = pair['drug_name'].map(drug_total)
    pair['rx_total']   = pair['reaction_term'].map(rx_total)
    pair['b'] = pair['drug_total'] - pair['a']
    pair['c'] = pair['rx_total']   - pair['a']
    pair['d'] = N - pair['drug_total'] - pair['rx_total'] + pair['a']
    pair['PRR'] = (pair['a']/(pair['a']+pair['b'])) / (pair['c']/(pair['c']+pair['d']).replace(0,np.nan))
    pair['ROR'] = (pair['a']*pair['d']) / (pair['b']*pair['c']).replace(0,np.nan)
    return pair

if proc.exists():
    sig = prr_ror(df).query('a >= 3 and PRR > 2 and ROR > 2').sort_values('PRR', ascending=False)
    print(f'{len(sig):,} pairs cross signal threshold')
    sig.head(20)

## 4. Cross-reference detected signals vs official FDA warnings

In [ ]:
warn = pd.read_csv(ROOT / 'data' / 'reference' / 'fda_warnings.csv', parse_dates=['warning_date'])
warn['drug_name']     = warn['drug_name'].str.upper()
warn['reaction_term'] = warn['reaction_term'].str.upper()
warn.head()

In [ ]:
if proc.exists():
    matched = sig.merge(warn, on=['drug_name','reaction_term'], how='inner')
    print(f'Matched {len(matched)} of {len(warn)} known FDA warnings in this sample')
    matched[['drug_name','reaction_term','a','PRR','ROR','warning_date','warning_type']]

## 5. Visualise top signals + backtest result

In [ ]:
if proc.exists():
    top = sig.head(20)
    fig, ax = plt.subplots(figsize=(8,6))
    ax.barh(top['drug_name'] + ' / ' + top['reaction_term'], top['PRR'])
    ax.set_xlabel('PRR')
    ax.set_title('Top 20 signals by PRR (sample)')
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

In [ ]:
report = ROOT / 'data' / 'processed' / 'backtest_report.json'
if report.exists():
    print(json.dumps(json.loads(report.read_text()), indent=2))
else:
    print('Run `python ml/evaluate.py` first.')